#  Spaceship Titanic - Competition Solution

**Kaggle Competition**: [Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic)

**Best Score**: 80.90% (Public Leaderboard)

## Approach Summary

This notebook documents the journey to achieve ~81% accuracy on the Spaceship Titanic competition:

1. **Feature Engineering** - Group-aware imputation, cabin features, spending patterns
2. **Model Selection** - Tested LightGBM, XGBoost, CatBoost and ensembles
3. **Regularization** - Strong regularization to reduce train-test gap
4. **Seed Averaging** - Multiple random seeds to reduce variance

**Key Insight**: CatBoost with seed averaging performed best, generalizing well from CV to public LB.

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
train.head()

## 2. Data Exploration

In [ ]:
# Check target distribution
print("Target Distribution:")
print(train['Transported'].value_counts(normalize=True))

# Missing values
print("\nMissing Values:")
print(train.isnull().sum())

## 3. Feature Engineering

Key features extracted:
- **Group features**: From PassengerId (group traveling together)
- **Cabin features**: Deck, CabinNum, Side from Cabin column
- **Spending features**: Total spend, log transforms, spending patterns
- **Derived features**: IsChild, AgeBin, CabinNumEven

In [ ]:
# Save IDs for submission
test_ids = test['PassengerId'].copy()

# Target
y = train["Transported"].astype(int).values

# Combine train and test for consistent processing
train["is_train"] = 1
test["is_train"] = 0
full = pd.concat([train, test], ignore_index=True)

print(f"Combined data shape: {full.shape}")

In [ ]:
# === GROUP FEATURES ===
# PassengerId format: GGGG_PP (Group_PersonInGroup)
pid_split = full["PassengerId"].str.split("_", expand=True)
full["Group"] = pid_split[0].astype(int)
full["GroupSize"] = full.groupby("Group")["PassengerId"].transform("count")
full["IsAlone"] = (full["GroupSize"] == 1).astype(int)

print("Group Size Distribution:")
print(full["GroupSize"].value_counts().head(10))

In [ ]:
# === CABIN FEATURES ===
# Cabin format: Deck/Num/Side (e.g., B/0/P)
cabin_split = full["Cabin"].str.split("/", expand=True)
full["Deck"] = cabin_split[0]
full["CabinNum"] = pd.to_numeric(cabin_split[1], errors="coerce")
full["Side"] = cabin_split[2]

print("Deck Distribution:")
print(full["Deck"].value_counts())

In [ ]:
# === SPENDING FEATURES ===
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
full["TotalSpend"] = full[spend_cols].sum(axis=1)
full["HasSpend"] = (full["TotalSpend"] > 0).astype(int)
full["NumAmenitiesUsed"] = full[spend_cols].gt(0).sum(axis=1)

print("Spending Stats:")
print(full["TotalSpend"].describe())

## 4. Missing Value Imputation

**Key Strategy**: Group-aware imputation
- People in the same group often share characteristics (HomePlanet, Destination, etc.)
- Use group mode for categorical, group median for numerical

In [ ]:
# Group-aware imputation for categorical columns
group_mode_cols = ["HomePlanet", "Destination", "Deck", "Side", "CryoSleep", "VIP"]
for col in group_mode_cols:
    full[col] = full.groupby("Group")[col].transform(
        lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
    )

# Group-aware imputation for numerical columns
for col in ["Age", "CabinNum"]:
    full[col] = full.groupby("Group")[col].transform(lambda x: x.fillna(x.median()))
    full[col] = full[col].fillna(full[col].median())  # Fallback to global median

# Fill spending with 0 (no spending)
for col in spend_cols:
    full[col] = full[col].fillna(0)

# CryoSleep logic: if in cryosleep, no spending possible
cs_na = full["CryoSleep"].isna()
full.loc[cs_na & (full["TotalSpend"] == 0), "CryoSleep"] = True
full.loc[cs_na & (full["TotalSpend"] > 0), "CryoSleep"] = False
full["CryoSleep"] = full["CryoSleep"].fillna(False)
full["VIP"] = full["VIP"].fillna(False)

# Remaining categorical with mode
for col in ["HomePlanet", "Destination", "Deck", "Side"]:
    full[col] = full[col].fillna(full[col].mode()[0])

print("Missing values after imputation:")
print(full.isnull().sum().sum())

## 5. Derived Features

In [ ]:
# Age-based features
full["IsChild"] = (full["Age"] < 18).astype(int)
full["AgeBin"] = pd.cut(full["Age"], bins=[-1, 12, 18, 25, 40, 60, 200], labels=False).astype(int)

# Cabin position feature (even/odd pattern observed in data)
full["CabinNumEven"] = (full["CabinNum"] % 2 == 0).astype(int)

# Log transforms for spending (reduces skewness)
full["LogTotalSpend"] = np.log1p(full["TotalSpend"])
for col in spend_cols:
    full["Log_" + col] = np.log1p(full[col])

print("Features created successfully!")

## 6. Encoding and Data Preparation

In [ ]:
# Drop columns not needed for modeling
full_features = full.drop(columns=["Cabin", "Name", "Group"])

# Encode categorical columns
cat_cols = full_features.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    full_features[col] = LabelEncoder().fit_transform(full_features[col].astype(str))

# Convert boolean to int
bool_cols = full_features.select_dtypes(include="bool").columns.tolist()
for col in bool_cols:
    full_features[col] = full_features[col].astype(int)

# Prepare feature columns
feature_cols = [c for c in full_features.columns if c not in ["Transported", "is_train", "PassengerId"]]

# Split back to train and test
train_processed = full_features[full_features["is_train"] == 1].copy()
test_processed = full_features[full_features["is_train"] == 0].copy()

X = train_processed[feature_cols].astype(float)
X_test = test_processed[feature_cols].astype(float)

print(f"Number of features: {len(feature_cols)}")
print(f"Feature list: {feature_cols}")

## 7. Model Training - CatBoost with Seed Averaging

**Why CatBoost?**
- Best generalization among tested models (LightGBM, XGBoost, CatBoost)
- Smallest gap between CV score and public LB score

**Why Seed Averaging?**
- Reduces variance in predictions
- Each seed produces slightly different models
- Averaging smooths out random fluctuations

In [ ]:
# Multiple seeds for averaging
seeds = [42, 123, 456, 789, 2024, 1337, 7777, 8888, 9999, 13]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_test_proba = []
all_oof_proba = []

print(f"Training with {len(seeds)} different seeds...")
print("="*50)

for seed in seeds:
    oof_proba = np.zeros(len(X))
    test_proba = np.zeros(len(X_test))
    
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y), 1):
        X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
        X_val, y_val = X.iloc[val_idx], y[val_idx]
        
        model = CatBoostClassifier(
            n_estimators=500,
            learning_rate=0.03,
            depth=5,
            subsample=0.6,
            l2_leaf_reg=3.0,
            min_data_in_leaf=40,
            random_state=seed,
            verbose=0
        )
        model.fit(X_tr, y_tr)
        oof_proba[val_idx] = model.predict_proba(X_val)[:, 1]
        test_proba += model.predict_proba(X_test)[:, 1] / cv.n_splits
    
    cv_acc = accuracy_score(y, (oof_proba >= 0.5).astype(int))
    print(f"Seed {seed}: CV = {cv_acc:.4f} ({cv_acc*100:.2f}%)")
    
    all_oof_proba.append(oof_proba)
    all_test_proba.append(test_proba)

print("="*50)

In [ ]:
# Average predictions across all seeds
avg_oof = np.mean(all_oof_proba, axis=0)
avg_test = np.mean(all_test_proba, axis=0)

# Calculate averaged CV score
avg_cv = accuracy_score(y, (avg_oof >= 0.5).astype(int))
print(f"\nAveraged CV Score: {avg_cv:.4f} ({avg_cv*100:.2f}%)")

## 8. Results Summary

| Approach | CV Score | Kaggle LB |
|----------|----------|----------|
| LightGBM (baseline) | 81.02% | 80.24% |
| XGBoost | 81.04% | - |
| CatBoost (single seed) | 81.28% | **80.83%** |
| CatBoost (5 seeds avg) | 81.38% | 80.90% |
| CatBoost (10 seeds avg) | 81.26% | **~81%** |

**Key Observations**:
- CatBoost generalizes better than LightGBM/XGBoost
- Seed averaging reduces variance and improves LB score
- Strong regularization is crucial to avoid overfitting

## 9. Create Submission

In [ ]:
# Create submission with threshold 0.5
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": (avg_test >= 0.5).astype(bool)
})

submission.to_csv("submission.csv", index=False)
print(f"Submission saved with {len(submission)} predictions")
submission.head()

In [ ]:
# Check prediction distribution
print("Prediction Distribution:")
print(submission["Transported"].value_counts(normalize=True))

## 10. Lessons Learned

1. **Group-aware imputation** is crucial - passengers traveling together share characteristics
2. **CryoSleep** is a strong predictor - people in cryosleep cannot spend money
3. **Regularization matters** - prevent overfitting with proper constraints
4. **Seed averaging** - simple technique that consistently improves scores
5. **Model selection** - CatBoost generalized better for this dataset

---
*Notebook created for Spaceship Titanic Kaggle Competition*